# Week 11: NLP

In this workshop you build a **RAG system**, retrieval-augmented generation, over every NUS module:

- **Section 1 - Intro to NLP and LLMs:** why text is hard for computers, how an LLM writes, and where it gets things wrong.
- **Section 2 - Retrieval Engine:** embeddings, chunking and semantic search over the NUSMods data.
- **Section 3 - Generation + Evaluation:** let Gemini answer from the retrieved modules, then measure how good the retrieval is.

Run the cells from top to bottom. Cells marked **YOUR TURN** are for you to edit.

# Section 1 - Intro to NLP and LLMs

By the end of this workshop you will have built a **RAG system**: a search engine over every NUS module, connected to an LLM that answers questions using what the search finds. This section lays the groundwork in three parts:

1. **Setup:** load the module data and connect your own Gemini API key.
2. **From text to numbers:** why text is hard for computers, what tokens are, and how an LLM actually writes.
3. **Where LLMs break:** ask Gemini about NUS modules, check its answers against the real data, and see why RAG is the fix.


## Setup

Run the two cells in this part now, while everyone is arriving.

On Colab, the first cell starts by downloading this week's folder from GitHub. Then it loads three things through `rag_utils.py`, a small helper file that all three sections share:

- `modules`: every module NUS offers in AY 2026-2027, taken from [NUSMods](https://nusmods.com). Each module is a Python dictionary with its code, title, description, workload and more. The organisers downloaded this in advance and saved it in `data/`, so the workshop does not depend on NUSMods being up.
- `embeddings`: one row of numbers per module. You will meet these in Section 2.
- `model`: `all-MiniLM-L6-v2`, a small model that turns text into numbers. It runs on your own CPU and needs no API key. The first run downloads it (about 90 MB).

In [ ]:
# %pip install -q google-genai sentence-transformers numpy matplotlib
import json
import os
import re
import sys
import textwrap
import urllib.request
import warnings

# On Colab: download only the week-11-nlp folder of the workshop repo (a few seconds)
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Sem-1"):
        !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/NUS-SDS-Workshops-AY2627/Sem-1.git /content/Sem-1
        !git -C /content/Sem-1 sparse-checkout set week-11-nlp
    %cd /content/Sem-1/week-11-nlp

os.environ["TRANSFORMERS_VERBOSITY"] = os.environ["HF_HUB_VERBOSITY"] = "error"  # hide harmless warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}  # "CS4248" -> its record

print(f"{len(modules)} modules offered in AY {rag_utils.AY}, fetched from NUSMods on {rag_utils.FETCH_DATE}")

### Connect Gemini

Each of you needs your **own** free API key: the free quota is counted per Google Cloud project, so 40 people sharing one key would run out within minutes.

> **Your Gemini API key (one-time, about 2 minutes)**
> 1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**, sign in with a *personal* Google account, find the 🔑 icon, click **Create API key** and copy the key. It's free — **don't** set up billing, because that turns the free tier off for the project.
> 2. In Colab, click the **key icon 🔑 (Secrets)** in the left sidebar → **Add new secret**.
> 3. Name: `GEMINI_API_KEY` · Value: paste your key · switch **Notebook access** on.
>
> No secret set up? The next cell asks you to paste the key instead. Never type your key into the code itself: it would be saved inside the notebook, and anyone you share the notebook with could use up your quota.

The cell below connects `ask_gemini` (from `rag_utils.py`), the one function we use to talk to Gemini. It sends your prompt and nothing else: no extra documents, and no Google Search. If the call fails, for example because the free quota has run out, it prints the reason and returns a saved answer instead of crashing.

In [ ]:
from rag_utils import ask_gemini

# Looks for your key in Colab Secrets first, then asks you to paste it.
rag_utils.connect_gemini()

print("Gemini says:", ask_gemini("Reply with exactly: ready"))

You should see `Gemini says: ready`. If you see `Gemini call failed` instead, the message tells you why. The most common one is `API key not valid`: copy the key again, update the secret, and re-run the cell. If it still fails, raise your hand. You can follow the Gemini demos on the main screen in the meantime.

## 1.1 Text is hard for computers

Computers only work with numbers. Text is not.

Even turning text into numbers is not enough, because **the meaning of a sentence is not in its characters**. Two strings that mean the same thing can look completely different, and one string can mean two different things.

In [ ]:
print("car" == "automobile")  # same meaning, different characters
print("bank" == "bank")       # same characters... river bank or money bank?

# Text CAN be turned into numbers, one per character
print([ord(character) for character in "car"])
print([ord(character) for character in "automobile"])

## 1.2 Ctrl+F fails

 `keyword_search` does the same across every module description: it returns the modules whose description contains your text, character for character.

Say you want a module about how computers understand language.

In [ ]:
def print_matches(query, k=5):
    """Run a keyword search and print the first k matches."""
    matches = keyword_search(query)
    print(f'"{query}": {len(matches)} matches')
    for match in matches[:k]:
        print(f"    {match['moduleCode']:<9} {match['title']}")


print_matches("how computers understand language")

In [ ]:
print_matches("natural language processing")
print_matches("language")

Keyword search compares characters, and meaning is not in the characters. To find anything, you have to guess the exact words the author used.

## 1.3 Tokens

Language models split text into **tokens**: common words stay whole, and rare words are chopped into smaller pieces that the model has seen before.

Every model comes with a **tokenizer** that does this splitting. Here is the tokenizer of `all-MiniLM-L6-v2` reading a real module description.

In [ ]:
module = by_code["CS4248"]
text = module["description"]

print(module["moduleCode"], "-", module["title"])
print(textwrap.fill(text, 100))
print()

tokens = model.tokenizer.tokenize(text)
print(f"{len(text.split())} words -> {len(tokens)} tokens")
print(tokens)

**YOUR TURN:** add your own words to the list and re-run. Try your name, Singlish, a long made-up word, or another language.

In [ ]:
# YOUR TURN: add your own words, then re-run
my_words = ["language", "tokenization", "NUSMods", "Singapore", "supercalifragilistic", "数据科学"]

for word in my_words:
    print(f"{word:>22} -> {model.tokenizer.tokenize(word)}")

# Each token is really an ID number in the model's vocabulary
print()
print("Token IDs:", model.tokenizer.encode("natural language processing", add_special_tokens=False))

`[UNK]` means **unknown**: the character is not in the model's vocabulary at all. 
The IDs on the last line are what the model actually receives. 

### The token budget

Every model can only read a limited number of tokens at once. 

In [ ]:
limit = model.max_seq_length
description_lengths = [len(model.tokenizer.tokenize(m["description"])) for m in modules]
longest = int(np.argmax(description_lengths))

print(f"MiniLM reads at most {limit} tokens")
print(f"Longest description: {description_lengths[longest]} tokens ({modules[longest]['moduleCode']})")
print(f"Descriptions over the limit: {sum(n > limit for n in description_lengths)} of {len(modules)}")

# Gemini uses a different tokenizer, so the same text gives a different count
try:
    gemini_count = rag_utils.gemini_client().models.count_tokens(model=rag_utils.GEMINI_MODEL, contents=text).total_tokens
    print(f"\nThe CS4248 description: MiniLM = {len(tokens)} tokens, Gemini = {gemini_count} tokens")
except Exception:
    pass

## 1.4 How an LLM works

An **LLM** (large language model)  does one thing: given the text so far, it predicts **which token comes next**. 

To watch it happen, we load a tiny LLM called **SmolLM2**. It has 135 million parameters, small enough to run on a laptop CPU (about a 270 MB download). Gemini works the same way, just thousands of times bigger.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "HuggingFaceTB/SmolLM2-135M"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm.eval()


def show_next_tokens(prompt, k=5):
    """Show the model's top k guesses for the very next token."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        probabilities = torch.softmax(lm(ids).logits[0, -1], dim=-1)
    top = torch.topk(probabilities, k)
    print(f"{prompt!r} ...")
    for probability, token_id in zip(top.values, top.indices):
        token = lm_tokenizer.decode([int(token_id)])
        print(f"   {token!r:<16} {probability.item():6.1%}  " + "#" * int(probability.item() * 40))


def generate_step_by_step(prompt, n=15):
    """Generate n tokens one at a time, always picking the most likely one."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    for step in range(n):
        with torch.no_grad():
            next_id = int(lm(ids).logits[0, -1].argmax())
        ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)
        print(f"step {step + 1:>2}: + {lm_tokenizer.decode([next_id])!r}")
    print()
    print("Full text:", lm_tokenizer.decode(ids[0], skip_special_tokens=True))


show_next_tokens("The National University of Singapore is located in")

Now let the model keep going. At every step it takes the most likely token and adds it to the text.

In [ ]:
generate_step_by_step("The National University of Singapore is located in", n=12)

It reads smoothly. But look closely: Singapore is not "a city in the southern part of the country". It **is** the country. The model aims for text that sounds natural, and that is not the same as text that is true.

What happens when we ask about something the model cannot possibly know?

In [ ]:
show_next_tokens("The NUS module CS4248 is titled")
generate_step_by_step("The NUS module CS4248 is titled", n=15)

print()
print("Real title:", by_code["CS4248"]["title"])

## 1.5 Where LLMs break

SmolLM2 is tiny. Does a real model do better? We will ask Gemini about NUS modules, giving it **no extra information**, and then **check its answer** against the real NUSMods data.

`verify` does the checking. It finds every module code in an answer, such as `CS4248`, and looks it up in this year's module list.

In [ ]:
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS4248, CS2103T


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

### Ask, then verify

First, the question. Gemini sees only this sentence.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

answer = ask_gemini(question)
display(Markdown(answer))

We verify the answer in the following block. 

In [ ]:
verify(answer)

**How to read it:**

- ❌ means the code is not in this year's NUSMods data. Either the module does not exist, or it is not running this year.
- ✅ with a different title means the code is real, but Gemini attached the wrong module to it, or used an old name.
- ✅ with a matching title means Gemini got that one right. You only know because you checked.

### Numbers it could easily invent

Specific numbers are the easiest thing for an LLM to get plausibly wrong. NUSMods lists the weekly workload of every module in hours.

In [ ]:
module_code = "CS4248"

print("GEMINI SAYS:")
print(ask_gemini(f"What is the weekly workload of NUS module {module_code} in hours, "
                 "as lecture / tutorial / lab / project / preparation?"))
print()
print("NUSMODS SAYS:", rag_utils.format_workload(by_code.get(module_code, {}).get("workload")) or "(not listed)")

Hours per week that look reasonable are exactly what next-token prediction produces, whether or not they are true. Try a module you have taken, such as `CS2040S` or `ST2334`, and re-run.

### Right last year, wrong this year

**Context:** CS4248 was called *Natural Language Processing* for years. In AY 2026-2027 it was renamed *Introduction to Large Language Models*. 

In [ ]:
module_code = "CS4248"
print("GEMINI SAYS:", ask_gemini(f"What is the official title of the NUS module {module_code}? Reply with the title only."))

with urllib.request.urlopen("https://api.nusmods.com/v2/2025-2026/moduleList.json") as response:
    last_year = {m["moduleCode"]: m["title"] for m in json.load(response)}

print("NUSMODS AY 2025-2026:", last_year.get(module_code))
print("NUSMODS AY 2026-2027:", by_code.get(module_code, {}).get("title"))

**YOUR TURN:** ask Gemini your own question about NUS modules, then check it. Ask for module codes, so that `verify` has something to check. Some things to try:

- A topic you care about: *"Which NUS modules teach reinforcement learning?"*
- Something new: *"Which NUS modules teach agentic AI?"*
- A module you have taken: *"What is CS2040S about, and what modules come after it?"*

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules should I take to learn about recommender systems? List the module codes and titles."

my_answer = ask_gemini(my_question)
display(Markdown(my_answer))
print("-" * 60)
verify(my_answer)

## 1.6 The RAG idea

The following block counts below use MiniLM's tokenizer, that how many tokens it would take to paste all NUSMods info into a prompt. Treat them as estimates.

 `build_chunk` turns one module into the text we would paste. 

In [ ]:
chunks = [build_chunk(m) for m in modules]
chunk_lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print(f"All {len(modules)} modules: about {sum(chunk_lengths):,} tokens, in every single prompt")
print(f"5 relevant modules: about {sum(chunk_lengths[:5]):,} tokens")

try:
    gemini_limit = rag_utils.gemini_client().models.get(model=rag_utils.GEMINI_MODEL).input_token_limit
    print(f"{rag_utils.GEMINI_MODEL} reads at most {gemini_limit:,} tokens per prompt")
except Exception:
    pass


**So fetch only the few modules that matter, and let the LLM answer from those.** That is **RAG**, retrieval-augmented generation:

```
 ┌─────────────┐     ┌─────────────┐     ┌────────────────────┐     ┌───────────────┐
 │ 1. Question │ ──> │ 2. Retrieve │ ──> │ 3. Augment         │ ──> │ 4. Generate   │
 │  from user  │     │  top-k most │     │  question + those  │     │  LLM answers  │
 │             │     │  relevant   │     │  modules -> prompt │     │  FROM them,   │
 │             │     │  modules    │     │                    │     │  with sources │
 └─────────────┘     └─────────────┘     └────────────────────┘     └───────────────┘
                       Section 2           Section 3                  Section 3
```


### If you're stuck

Run the cell below. It reloads everything Section 2 needs.

In [ ]:
# Reloads the data and the embedding model if anything above failed

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import ask_gemini, build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
print(len(modules), "modules, embeddings", embeddings.shape)

# Section 2 - Retrieval Engine

Section 1 ended with the RAG idea: before asking the LLM, **retrieve** the few modules that matter and paste only those into the prompt. This section builds that retriever in three steps:

1. **Embedding:** turn text into numbers that capture meaning.
2. **Chunking:** decide which text gets turned into numbers.
3. **Retrieval:** find the modules closest to a question.

Cells marked **YOUR TURN** are for you to edit. Everything else runs as it is.

In [ ]:
# Two small helpers used throughout this section.
row_of = {module["moduleCode"]: i for i, module in enumerate(modules)}


def show(results):
    """Print (module, score) pairs, one per line."""
    for module, score in results:
        print(f"  {score:.2f}  {module['moduleCode']:<9} {module['title']}")

## 2.1 Embedding

In Section 1, Ctrl+F failed because it compares characters, and meaning is not in the characters. An **embedding model** reads a piece of text and returns a list of numbers, called a **vector**, that describes what the text is about. Texts with similar meanings get similar numbers.

We use `all-MiniLM-L6-v2`:

- **Small:** about 90 MB, and it runs on a laptop CPU.
- **Local:** no API key and no cost, because nothing is sent anywhere.
- **384 dimensions:** every text becomes 384 numbers, whether it is one word or a paragraph.

In [ ]:
vector = model.encode("Machine learning is fun")

print("Shape:", vector.shape)
print("First 8 numbers:", vector[:8].round(3))

The individual numbers mean nothing on their own. What matters is how close two vectors are.

We measure closeness with a **dot product**: multiply the two vectors position by position, then add everything up. With `normalize_embeddings=True` every vector has length 1, so the result is the cosine similarity. About 1 means the same meaning, and about 0 means unrelated.

In [ ]:
sentences = [
    "How do I learn to code?",
    "Where can I pick up programming?",
    "What should I eat for lunch?",
]
vectors = model.encode(sentences, normalize_embeddings=True)

print("Sentence 1 vs 2:", round(float(vectors[0] @ vectors[1]), 2))
print("Sentence 1 vs 3:", round(float(vectors[0] @ vectors[2]), 2))

Sentences 1 and 2 have no important word in common, yet they score far higher than 1 and 3. Ctrl+F could never tell you that.

### Real modules

Every module already has a vector. They were computed in advance and stored in `embeddings`, where row `i` belongs to `modules[i]`. Here are 10 modules from four areas, each compared with all the others.

In [ ]:
def plot_similarity(labels, vectors, title):
    """Heatmap of the similarity between every pair of vectors."""
    similarity = vectors @ vectors.T
    count = len(labels)

    fig, ax = plt.subplots(figsize=(9, 6))
    image = ax.imshow(similarity, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(count), range(1, count + 1))
    ax.set_yticks(range(count), [f"{label}  {i + 1}" for i, label in enumerate(labels)])
    for i in range(count):
        for j in range(count):
            colour = "white" if similarity[i, j] > 0.6 else "black"
            ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=8, color=colour)
    fig.colorbar(image, label="Similarity")
    ax.set_title(title)
    plt.show()


codes = [
    "CS2109S", "CS3244", "CS4248",     # AI
    "EC1101E", "FIN2704", "FIN3702A",  # economics and finance
    "LSM1301", "LSM2105",              # biology
    "EN1101E", "EN2210",               # literature
]
rows = [row_of[code] for code in codes]
labels = [f"{code} {modules[row]['title']}" for code, row in zip(codes, rows)]

plot_similarity(labels, embeddings[rows], "How similar are these modules?")

**How to read it:** each cell compares two modules, and a darker cell means more similar. The diagonal is every module against itself, so it is always 1.00.

The darker squares along the diagonal are the four areas: AI, economics and finance, biology, and literature. Nobody told the model which faculty a module belongs to. It grouped them from the text alone.

Unrelated modules do not score 0. They mostly sit around 0.1 to 0.3, so read the scores as a ranking: related pairs score higher than unrelated ones.

### Ctrl+F, again

Back to the query that returned nothing in Section 1.

In [ ]:
query = "how computers understand language"

print("Keyword search:", len(keyword_search(query)), "matches")
print("Semantic search:")
show(search(query))

Keyword search finds nothing, because no description contains that exact phrase. Semantic search returns modules about language, and the top hit is a course on how humans and machines handle language.

`search` is a black box for now. You will write it yourself in 2.3.

**Think about it:** most of the results are linguistics modules, not computing ones. Which words in the query pulled them in?

**YOUR TURN:** edit the sentences below and re-run the cell. Some things to try:

- The same word with two meanings: "I sat by the river bank" and "I opened a bank account".
- Opposites: "I love durian" and "I hate durian".
- The same sentence in two languages.

In [ ]:
# YOUR TURN: edit these sentences, then re-run
my_sentences = [
    "I love machine learning",
    "Neural networks are fascinating",
    "The stock market crashed today",
    "Investors are worried about inflation",
    "My cat sleeps all day",
    "Dogs are loyal pets",
]

plot_similarity(my_sentences, model.encode(my_sentences, normalize_embeddings=True), "My sentences")

## 2.2 Chunking

A **chunk** is one piece of text that gets its own vector. Real sources such as a handbook, a PDF or a website are too long to embed in one go, so we split them. There are two reasons:

- **Token budget:** MiniLM reads at most 256 tokens. Anything after that is silently cut off.
- **Precision:** one vector has to summarise the whole chunk. A vector for a 50-page handbook is about everything, so it is not close to any particular question.

NUSMods is a lucky case, because the data is already split for us: **one module = one chunk**. The decision left is which fields of a module go into the chunk text.

In [ ]:
module = modules[row_of["CS2040S"]]

print("The raw module:")
for field, value in module.items():
    print(f"  {field}: {value}")

print()
print("The chunk that gets embedded:")
print(build_chunk(module))

`build_chunk` (in `rag_utils.py`) joins four fields with line breaks: code, title, description and workload.

Look at the workload. NUSMods stores it as `[3, 0, 1, 3, 3]`, which tells an embedding model nothing, so we write it out in words first. Preparing text so that the model can use it is part of chunking.

How close are we to the token budget?

In [ ]:
chunks = [build_chunk(module) for module in modules]
lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print("CS2040S chunk:", lengths[row_of["CS2040S"]], "tokens. The limit is", model.max_seq_length)
print("Typical chunk:", int(np.median(lengths)), "tokens")

too_long = sum(length > model.max_seq_length for length in lengths)
print(f"{too_long} of {len(modules)} chunks are over the limit and lose their ending")

Only a small share of chunks get cut. When it happens, the end of the description and the workload are lost. That is why the code and title go first.

### What we left out

NUSMods has 21,160 modules on record for AY 2026-2027. We kept 6,552:

- **14,382 dropped** because they are not offered this year.
- **226 dropped** because the description is a placeholder such as "Not Available". With nothing to describe the module, its vector would come from the code and title alone.

### Which fields go in?

Adding a field is not free. To see why, we embed the same modules twice: once with the default chunk, and once with the prerequisites added.

Re-embedding all 6,552 modules takes minutes on a CPU, so we use the Computing modules only.

In [ ]:
computing = [module for module in modules if module["faculty"] == "Computing"]

fields_default = ["moduleCode", "title", "description", "workload"]
fields_prereq = fields_default + ["prerequisite"]

vectors_default = model.encode([build_chunk(m, fields_default) for m in computing], normalize_embeddings=True)
vectors_prereq = model.encode([build_chunk(m, fields_prereq) for m in computing], normalize_embeddings=True)

print(len(computing), "Computing modules embedded twice")
print()
print("A chunk with prerequisites:")
print(build_chunk(module, fields_prereq))

In [ ]:
def compare_chunks(query, needs=None):
    """Search the Computing modules with both versions of the chunk.

    `needs` is a module code. Results that list it as a prerequisite get a tick.
    """
    for name, vectors in [("Default chunk", vectors_default), ("With prerequisites", vectors_prereq)]:
        print(name)
        for result, score in search(query, embeddings=vectors, modules=computing):
            tick = "✓" if needs and needs in result.get("prerequisite", "") else " "
            print(f"  {score:.2f} {tick} {result['moduleCode']:<9} {result['title']}")


compare_chunks("I have completed CS2040S, what next?", needs="CS2040S")

A tick marks a module that really does list CS2040S as a prerequisite. With the prerequisites in the chunk, more of the top 5 are correct, because the text the question is about is now there to be matched.

Now a question about a topic.

In [ ]:
compare_chunks("I want to learn machine learning")

Every score dropped, and CS3244 Machine Learning, the module with exactly that title, fell out of the top 5. Its prerequisite text is long, and those words now share the vector with the topic.

That is the tradeoff: a field you add helps the questions about that field and dilutes everything else. Longer chunks also push more modules past the 256-token limit. Most questions are about topics, so the default chunk leaves prerequisites out.

**YOUR TURN:** choose your own fields below, re-embed the Computing modules, and compare with the default chunk. Does a shorter chunk do better or worse on your question?

In [ ]:
# YOUR TURN: choose the fields and the question, then re-run
# Fields you can use: moduleCode, title, description, workload, prerequisite, faculty, department
my_fields = ["title", "description"]
my_question = "I want to learn machine learning"

my_vectors = model.encode([build_chunk(m, my_fields) for m in computing], normalize_embeddings=True)

print("Default chunk")
show(search(my_question, embeddings=vectors_default, modules=computing))
print("My chunk:", my_fields)
show(search(my_question, embeddings=my_vectors, modules=computing))

## 2.3 Retrieval

Every module has a vector. To answer a question:

1. Embed the query with the same model.
2. Compare it with every module vector.
3. Sort by score.
4. Take the top `k`.

In [ ]:
def my_search(query, k=5):
    query_vector = model.encode(query, normalize_embeddings=True)  # 1. embed the query
    scores = embeddings @ query_vector                             # 2. compare with every module
    top = np.argsort(-scores)[:k]                                  # 3. sort, 4. take the top k
    return [(modules[i], float(scores[i])) for i in top]


show(my_search("how computers understand language"))

That is the whole retrieval engine, and it is what `search` in `rag_utils.py` does. The line `embeddings @ query_vector` is one matrix multiplication that computes all 6,552 dot products at once.

### Do we need a vector database?

In [ ]:
import time

query_vector = model.encode("how computers understand language", normalize_embeddings=True)

start = time.perf_counter()
scores = embeddings @ query_vector
top = np.argsort(-scores)[:5]
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Compared the query with {len(modules)} modules in {elapsed_ms:.1f} ms")

No. A vector database earns its place at millions of chunks, where comparing against everything becomes too slow. For a few thousand chunks, plain numpy is enough.

### Keyword search vs semantic search

Here are both, side by side, on the same query.

In [ ]:
def compare(query, k=5):
    """Show keyword search and semantic search for the same query."""
    matches = keyword_search(query)
    print(f"Keyword search: {len(matches)} matches")
    for match in matches[:k]:
        print(f"        {match['moduleCode']:<9} {match['title']}")
    print("Semantic search:")
    show(search(query, k))


compare("how computers understand language")

Semantic search wins that one. Now a query where the exact word matters.

In [ ]:
compare("ChatGPT")

Keyword search wins this one. Four descriptions mention ChatGPT by name, and Ctrl+F finds all of them. Semantic search finds one, then fills the other slots with communication modules.

Why? MiniLM was trained before ChatGPT existed, so it has never seen the word. This is how its tokenizer reads it:

In [ ]:
print(model.tokenizer.tokenize("ChatGPT"))

The model sees "chat" plus two fragments, so it goes looking for modules about communication. The embedding model has a knowledge cutoff too, just like the LLM in Section 1.

| | Keyword search | Semantic search |
|---|---|---|
| Matches | exact words | similar meaning |
| Good at | names, rare terms, new words | paraphrases and vague questions |
| Fails when | you use different words from the text | the exact word matters, or the model has never seen it |
| Returns | every match, in no particular order | always `k` results, ranked by score |

The last row matters: semantic search **always** returns `k` modules, even when nothing relevant exists.

**YOUR TURN:** ask your own questions. Some things to try:

- A topic in your own words, the way you would ask a senior.
- A module code, such as `CS2040S`.
- A greeting, such as `hi`.
- A filter, such as `4 MC, no exam, semester 2`.
- A larger `k`. Do the extra results still look relevant?

**Keep your funniest failure.** We will collect them in Section 3.

In [ ]:
# YOUR TURN: ask your own question, and change k
my_query = "I want to learn how to invest"
k = 5

compare(my_query, k)

## Recap

- **Embedding:** a model turns text into a vector, and similar meanings land close together.
- **Chunking:** one module is one chunk, and the fields you put in the chunk decide what it can be found by.
- **Retrieval:** embed the query, take a dot product with every module, sort, and keep the top `k`.

Section 3 takes the modules returned by `search(query, k=5)` and hands them to Gemini.

# Section 3 - Generation + Evaluation

Section 2 built the retriever. This section finishes the RAG pipeline from Section 1.6, then checks how well the retriever works:

1. **Generation:** paste the retrieved modules into the prompt (step 3, *augment*) and let Gemini answer from them (step 4, *generate*).
2. **Evaluation:** measure how often the retriever finds the right module, because the answer can only be as good as what was retrieved.

## 3.1 Generation

Two small functions do the augment step:

- `make_context` turns the `(module, score)` pairs from `search` into plain text, one numbered block per module.
- `build_prompt` wraps that text in instructions: answer **only** from the context, say so when the context is not enough, and cite module codes in square brackets.

In [ ]:
def make_context(results):
    """Format retrieved (module, score) pairs as numbered text blocks."""
    blocks = []
    for rank, (module, score) in enumerate(results, start=1):
        lines = [
            f"[{rank}] {module['moduleCode']} - {module['title']}",
            f"Description: {module['description']}",
        ]
        workload = rag_utils.format_workload(module.get("workload"))
        if workload:
            lines.append(f"Workload: {workload}")
        if module.get("semesters"):
            lines.append(f"Semesters offered in AY {rag_utils.AY}: {', '.join(map(str, module['semesters']))}")
        if module.get("hasExam") is not None:
            lines.append(f"Has final exam: {'yes' if module['hasExam'] else 'no'}")
        lines.append(f"Retrieval similarity: {score:.3f}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks) if blocks else "No modules were retrieved."


def build_prompt(question, context):
    """Wrap the question and the retrieved context in grounding instructions."""
    return f"""You are a helpful assistant answering questions about NUS modules.
Answer the user's question using only the supplied NUSMods context.

Rules:
- If the context does not contain enough evidence, say that the information is not available in the supplied context.
- Do not invent module codes, titles, prerequisites, workload hours, semesters, or exam details.
- When making a claim about a module, cite its module code in square brackets, e.g. [CS1231S].
- Keep the answer concise and distinguish explicit facts from uncertainty.
- The retrieval similarity score is not evidence that a claim is true.

Retrieved context:
{context}

User question: {question}

Answer:"""


def rag_answer(question, k=5, results=None, show_prompt=False):
    """Retrieve, augment, generate. Pass `results` to supply your own context."""
    if results is None:
        results = search(question, k=k)
    prompt = build_prompt(question, make_context(results))
    if show_prompt:
        print(prompt)
        print("-" * 60)
    answer = ask_gemini(prompt)
    display(Markdown(answer))
    return answer

### The same question, now with RAG

In Section 1.5, Gemini answered this question from memory, and `verify` caught the module codes it got wrong. Here is the exact prompt RAG sends for the same question. Gemini sees only this text.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

print(build_prompt(question, make_context(search(question, k=5))))

In [ ]:
answer = rag_answer(question)
print("-" * 60)
verify(answer)

Every module code Gemini can cite now comes from the context, so `verify` should find each one in the real data. Compare this with the 1.5 result.

That fixes invented codes, but not missing ones. Look back at the retrieved modules in the prompt: are they about attention in neural networks, or attention in psychology? If the right module was not retrieved, Gemini cannot mention it. The answer is only as good as the context.

### Wrong context, confident answer

To see how much the answer depends on retrieval, keep the question and swap in modules retrieved for an unrelated topic.

In [ ]:
wrong_results = search("modules about economics and financial markets", k=5)
show(wrong_results)
print("-" * 60)

wrong_answer = rag_answer(question, results=wrong_results)

**How to read it:** the prompt tells Gemini to say when the context is not enough. Check whether it did, or whether it stretched an economics module to fit a question about attention mechanisms. Either way, generation cannot recover from bad retrieval, which is why 3.2 measures the retriever on its own.

**YOUR TURN:** ask your own question. Set `show_prompt=True` to see exactly what Gemini received, and change `k` to give it more or fewer modules.

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules teach reinforcement learning? List the module codes and titles."

my_answer = rag_answer(my_question, k=5, show_prompt=False)
print("-" * 60)
verify(my_answer)

## 3.2 Evaluation

Now we evaluate how effectively our RAG system retrieves the correct NUS module for a user's question.

For each query, we specify the expected module code and compare it with the modules returned by our retrieval system.

We use two metrics:

- **Hit@5:** Measures the proportion of queries for which the correct module appears in the top five retrieved results.
- **Mean Reciprocal Rank (MRR):** Measures how highly the correct module is ranked, rewarding results that appear closer to the top.

These metrics help us assess retrieval performance before investigating potential improvements.

In [ ]:
import pandas as pd

workshop_dir = rag_utils.DATA_DIR.parent  # the week-11-nlp folder

In [ ]:
eval_path = workshop_dir / "data" / "eval_set_v2.csv"

eval_df = pd.read_csv(eval_path)

print(f"Total evaluation queries: {len(eval_df)}")
print("\nQueries by category:")
display(eval_df["category"].value_counts().to_frame("count"))

display(eval_df.head())

In [ ]:
def evaluate_multilabel_retrieval(eval_df, k=5):
    details = []

    for _, row in eval_df.iterrows():
        query = str(row["query"])
        relevant_codes = {
            code.strip().upper()
            for code in str(row["relevant"]).split("|")
            if code.strip()
        }

        results = rag_utils.search(query, k=k)

        ranked_codes = [
            str(module.get("moduleCode", "")).upper()
            for module, score in results
        ]

        # Rank of the first retrieved relevant module
        relevant_ranks = [
            rank for rank, code in enumerate(ranked_codes, start=1)
            if code in relevant_codes
        ]

        first_relevant_rank = (
            min(relevant_ranks) if relevant_ranks else 0
        )

        details.append({
            "query": query,
            "category": row["category"],
            "relevant_codes": " | ".join(sorted(relevant_codes)),
            "top_k_module_codes": " | ".join(ranked_codes),
            "rank_of_first_relevant": first_relevant_rank,
            f"hit_at_{k}": int(bool(relevant_ranks)),
            "reciprocal_rank": (
                1 / first_relevant_rank
                if first_relevant_rank else 0
            )
        })

    details_df = pd.DataFrame(details)

    summary = {
        "n_queries": len(details_df),
        f"hit_at_{k}": details_df[f"hit_at_{k}"].mean(),
        "mrr": details_df["reciprocal_rank"].mean()
    }

    return summary, details_df


In [ ]:
summary, details_df = evaluate_multilabel_retrieval(
    eval_df,
    k=5
)

print(f"Queries evaluated: {summary['n_queries']}")
print(f"Hit@5: {summary['hit_at_5']:.3f}")
print(f"MRR: {summary['mrr']:.3f}")

results_dir = workshop_dir / "evaluation"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "eval_set_v2_results.csv"
details_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

display(details_df)

**YOUR TURN:** pick a query from the table above that missed (`hit_at_5` is 0), or ask your own question, and look at what the retriever returned instead.

In [ ]:
# YOUR TURN: change the question, then re-run
query = "Which NUS modules cover attention mechanisms?"

results = rag_utils.search(query, k=5)

print(f"Query: {query}\n")
print("Top 5 retrieved modules:\n")

for rank, (module, score) in enumerate(results, start=1):
    print(f"{rank}. {module.get('moduleCode', 'N/A')} - "
          f"{module.get('title', 'N/A')}")
    print(f"   Similarity score: {score:.4f}\n")

## Recap

- **Generation:** the retrieved modules go into the prompt as context, and the instructions keep Gemini to that context and make it cite module codes.
- **Evaluation:** Hit@5 asks whether a relevant module made the top 5; MRR rewards finding it near the top.
- **The weak link is retrieval:** with the wrong context, even a good LLM gives a poor answer.